In [ ]:
library(BSgenome.Oaries.NCBI.Oarv4.0)
library(motifbreakR)
library(TFBSTools)
library(JASPAR2024)
library(RSQLite)


In [ ]:
snp_bed <- data.frame(
    chr = rep("NC_019466.2", 8),
    start = c(
        14361049,
        14361086,
        14363010,
        14363054,
        14363169,
        14363191,
        14363214,
        14363236
    ),
    end = c(
        14361050,
        14361087,
        14363011,
        14363055,
        14363170,
        14363192,
        14363215,
        14363237
    ),
    name = c(
        "NC_019466.2:14361050:T:A",
        "NC_019466.2:14361087:C:T",
        "NC_019466.2:14363011:G:A",
        "NC_019466.2:14363055:T:C",
        "NC_019466.2:14363170:C:T",
        "NC_019466.2:14363192:C:T",
        "NC_019466.2:14363215:C:T",
        "NC_019466.2:14363237:G:A"
    )
)

snp_bed_file <- file.path(
    "/home/hnadmin/data/yangshuju/result/ovary/peak/motifbreakR",
    "sheep_8SNP_Oar_v4.bed"
)

write.table(
    snp_bed,
    snp_bed_file,
    sep = "\t",
    quote = FALSE,
    row.names = FALSE,
    col.names = FALSE
)

snp_bed

In [ ]:
snp_df <- data.frame(
    chr = rep("NC_019466.2", 8),
    pos = c(
        14361050,
        14361087,
        14363011,
        14363055,
        14363170,
        14363192,
        14363215,
        14363237
    ),
    REF = c(
        "T",
        "C",
        "G",
        "T",
        "C",
        "C",
        "C",
        "G"
    ),
    ALT = c(
        "A",
        "T",
        "A",
        "C",
        "T",
        "T",
        "T",
        "A"
    ),
    stringsAsFactors = FALSE
)

snpList <- GRanges(
    seqnames = snp_df$chr,
    ranges = IRanges(
        start = snp_df$pos,
        end   = snp_df$pos
    ),
    REF = snp_df$REF,
    ALT = snp_df$ALT
)

names(snpList) <- paste(
    snp_df$chr,
    snp_df$pos,
    snp_df$REF,
    snp_df$ALT,
    sep = ":"
)

strand(snpList) <- "*"

snpList


In [ ]:
# 保留人和小鼠来源的 JASPAR2024 motif
jaspar2024_mammal <- jaspar2024[
    grepl("^(Hsapiens|Mmusculus)-", names(jaspar2024))
]

In [ ]:
res<-motifbreakR(snpList = snpList, filterp = TRUE,
                       pwmList = jaspar2024_mammal,
                       threshold = 1e-4,
                       method = "ic",
                       bkg = c(A=0.25, C=0.25, G=0.25, T=0.25),
                       BPPARAM = BiocParallel::bpparam())

In [ ]:
res$SNP_id <- names(res)
options(ucscChromosomeNames = FALSE)
plotMB(res, 'NC_019466.2:14363237:G:A',
    effect = "strong"
)

In [ ]:
plot_dir <- file.path(outdir, "plotMB_each_SNP")

if (!dir.exists(plot_dir)) {
    dir.create(plot_dir, recursive = TRUE)
}

snp_ids <- unique(names(res))

for (snp in snp_ids) {

    cat("Plotting:", snp, "\n")

    safe_name <- gsub(":", "_", snp)

    # PDF
    pdf(
        file = file.path(
            plot_dir,
            paste0(safe_name, "_plotMB.pdf")
        ),
        width = 10,
        height = 8,
        useDingbats = FALSE
    )

    tryCatch(
        {
            plotMB(
                results = res,
                rsid = snp,
                effect = "strong"
            )
        },
        error = function(e) {
            message("Failed: ", snp)
            message("Error: ", e$message)
        }
    )

    dev.off()
}

In [ ]:
# ============================================================
# 1. motifbreakR res -> data.frame
#    避免 duplicate row.names
# ============================================================

res_tmp <- res

# 保存原始 SNP ID
snp_id <- names(res_tmp)

# 去掉 GRanges names，避免 as.data.frame() 使用重复 SNP ID 作为 row.names
names(res_tmp) <- NULL

# 转换
res_df <- as.data.frame(res_tmp)

# 加回 SNP ID
res_df$snp_id <- snp_id

# SNP ID 放到第一列
res_df <- res_df[, c(
    "snp_id",
    setdiff(colnames(res_df), "snp_id")
)]

# 清除 rownames
rownames(res_df) <- NULL

In [ ]:
tpm_file <- "/home/hnadmin/data/yangshuju/result/ovary/TPM_HTY_HY_result.txt"

tpm <- read.delim(
    tpm_file,
    header = TRUE,
    sep = "\t",
    stringsAsFactors = FALSE,
    check.names = FALSE
)

tpm <- tpm[, c(
    "Geneid",
    "HTY_mean_TPM",
    "HY_mean_TPM"
)]

# 防止 Geneid 重复
tpm <- tpm[!duplicated(tpm$Geneid), ]